In [2]:
import cv2
import numpy as np
import time
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn

# --- Configuration ---
model_path = "/home/dimple/Desktop/Model_Training/Training_Results/FasterRCNN/best_model_epoch_175.pth"
image_path = "/home/dimple/Desktop/Model_Training/Test_Images/IMG_0501_1_5_Red edge.tif"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- Load model ---
# Use the same architecture you trained
model = fasterrcnn_resnet50_fpn(weights=None, num_classes=3)  # Change num_classes as per your training
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# --- Load and preprocess image ---
image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
if image is None:
    raise FileNotFoundError(f"Image not found: {image_path}")

# Convert 16-bit to 8-bit if needed
if image.dtype == np.uint16:
    image = (image / 256).astype(np.uint8)

# Convert grayscale to RGB
if len(image.shape) == 2:
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
elif image.shape[2] == 1:
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

# Convert image to tensor format
image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
image_tensor = image_tensor.unsqueeze(0).to(device)

# --- Warm-up (for consistent GPU timing) ---
with torch.no_grad():
    _ = model(image_tensor)

# --- Time inference for 1 image ---
start = time.time()
with torch.no_grad():
    outputs = model(image_tensor)
end = time.time()

# --- Report result ---
print(f"\nInference time for 1 image using Faster-RCNN: {end - start:.4f} seconds")
print("Number of detections:", len(outputs[0]["boxes"]))

/tmp/ipykernel_9268/362633474.py:15: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))



Inference time for 1 image using Faster-RCNN: 1.0620 seconds
Number of detections: 5
